# 2. Clustering – Mall Customer Segmentation
**Dataset:** `Mall_Customers.csv` (200 rows) | **Task:** Unsupervised customer segmentation  
**Algorithms (4):** K-Means, Hierarchical (Agglomerative), DBSCAN, Gaussian Mixture Model

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from scipy.cluster.hierarchy import dendrogram, linkage
sns.set_style('whitegrid')

## 1. Load & explore

In [ ]:
df = pd.read_csv('data/Mall_Customers.csv')
display(df.head()); print(df.shape); print(df.isnull().sum()); display(df.describe())
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, c in zip(ax, ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']):
    sns.histplot(df[c], kde=True, ax=a); a.set_title(c)
plt.tight_layout(); plt.show()
sns.pairplot(df.drop('CustomerID', axis=1), hue='Gender'); plt.show()

## 2. Feature selection & scaling
We cluster on **Annual Income** and **Spending Score**, the two features that define shopping behaviour.

In [ ]:
X = df[['Annual Income (k$)', 'Spending Score (1-100)']]
Xs = StandardScaler().fit_transform(X)

## 3. Choosing k (Elbow + Silhouette)

In [ ]:
ks = range(2, 11)
inertia = [KMeans(n_clusters=k, n_init=10, random_state=42).fit(Xs).inertia_ for k in ks]
sil = [silhouette_score(Xs, KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(Xs)) for k in ks]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(ks, inertia, 'o-'); ax[0].set_title('Elbow method'); ax[0].set_xlabel('k'); ax[0].set_ylabel('Inertia')
ax[1].plot(ks, sil, 'o-', color='green'); ax[1].set_title('Silhouette score'); ax[1].set_xlabel('k')
plt.tight_layout(); plt.show()
best_k = list(ks)[int(np.argmax(sil))]
print('Best k by silhouette:', best_k)
K = 5   # elbow + business interpretation (5 well-known segments)

## 4. Train the 4 clustering algorithms

In [ ]:
labels = {}
labels['K-Means'] = KMeans(n_clusters=K, n_init=10, random_state=42).fit_predict(Xs)
labels['Hierarchical (Ward)'] = AgglomerativeClustering(n_clusters=K, linkage='ward').fit_predict(Xs)
labels['DBSCAN'] = DBSCAN(eps=0.4, min_samples=5).fit_predict(Xs)
labels['Gaussian Mixture'] = GaussianMixture(n_components=K, random_state=42).fit(Xs).predict(Xs)

plt.figure(figsize=(10, 4))
dendrogram(linkage(Xs, method='ward'), truncate_mode='lastp', p=20); plt.title('Dendrogram (Ward)'); plt.show()

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
for ax, (name, lab) in zip(axes.ravel(), labels.items()):
    sns.scatterplot(x=X.iloc[:, 0], y=X.iloc[:, 1], hue=lab, palette='tab10', ax=ax, legend='full')
    ax.set_title(f'{name}  (clusters found: {len(set(lab)) - (1 if -1 in lab else 0)})')
plt.tight_layout(); plt.show()

## 5. Evaluation (internal metrics – no ground truth exists)

In [ ]:
rows = []
for name, lab in labels.items():
    mask = lab != -1                       # ignore DBSCAN noise points for the metrics
    n_clusters = len(set(lab[mask]))
    if n_clusters > 1:
        rows.append({'Algorithm': name, 'Clusters': n_clusters, 'Noise pts': int((~mask).sum()),
                     'Silhouette (higher better)': silhouette_score(Xs[mask], lab[mask]),
                     'Davies-Bouldin (lower better)': davies_bouldin_score(Xs[mask], lab[mask]),
                     'Calinski-Harabasz (higher better)': calinski_harabasz_score(Xs[mask], lab[mask])})
res = pd.DataFrame(rows).set_index('Algorithm').round(4)
display(res)

## 6. Interpreting the K-Means segments

In [ ]:
df['Cluster'] = labels['K-Means']
profile = df.groupby('Cluster')[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']].mean().round(1)
profile['Customers'] = df['Cluster'].value_counts().sort_index()
display(profile)

## 7. Conclusion
- K-Means, Hierarchical and GMM all recover essentially the same **5 customer segments**: low income/low spend, low income/high spend, mid income/mid spend, high income/low spend, high income/high spend.
- DBSCAN finds dense regions but labels some customers as noise and is sensitive to `eps`.
- K-Means (or GMM) is the best choice here – highest silhouette, simple to interpret, and gives actionable marketing segments (e.g. the *high income / high spending* group is the target for premium offers).